# 02 · Cleaning

**Input:** `data/petitions.jsonl` — the snapshot of 25 Sep 2026 (128,077 petitions).

**Output** (in `data/processed/`, not in git):

| File | Content |
|---|---|
| `petitions.parquet` | 127,191 petitions: 7-class label, cleaned text, time columns |
| `out_of_scope.parquet` | the 886 dropped petitions (fake-name, operational, foi), kept for the out-of-scope test in week 9 |
| `gold_pairs.parquet` | one row per (duplicate petition, published petition it duplicates) |

Steps: 1 load (allowed fields only) → 2 labels → 3 decision time → 4 text cleaning → 5 gold duplicate pairs → 6 leakage test → 7 save.

The cleaning functions are in `src/preprocess.py`, so the final pipeline cleans a new petition in exactly the same way.

In [1]:
import collections, hashlib, json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent                      # this notebook runs inside notebooks/
sys.path.insert(0, str(ROOT / "src"))
from preprocess import (CLASSES, CLEAN_COLUMNS, MODEL_INPUTS, ANGLE_RE, URL_RE,
                        build_text, clean_text, make_label, linked_ids, leakage_check)

RAW = ROOT / "data" / "petitions.jsonl"
OUT = ROOT / "data" / "processed"
OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 100)

## 1. Load the snapshot

First check that the file is exactly the Milestone-1 snapshot (same SHA-256 hash), so every number below can be reproduced.

Then copy **only the allowed fields** into a table (an allow-list). Fields that are set *after* the moderation decision
(`signature_count`, `departments`, `committee_note`, `government_response`, `debate`, threshold dates, `updated_at`)
and `creator_name` (personal data) are never copied, so they cannot leak into a model by mistake.

- `action`, `background`, `additional_details` — the petition text, the only model input.
- `state` + `rejection_code` — the label.
- `rejection_details` — only to find the duplicate links in step 5; it is not saved.
- `opened_at`, `rejected_at`, `closed_at` — only for the time split and the candidate pools.
- `created_at` — submission time; only for the EDA (how long moderation takes). Never a feature.

In [2]:
SNAPSHOT_SHA256 = "499ad9e79f61cdb032d1b22a31a49634d6f175a68844e38f13e388ac816be519"
sha = hashlib.sha256(RAW.read_bytes()).hexdigest()
assert sha == SNAPSHOT_SHA256, "data/petitions.jsonl is not the 25 Sep 2026 snapshot"
print("Snapshot OK:", sha)

Snapshot OK: 499ad9e79f61cdb032d1b22a31a49634d6f175a68844e38f13e388ac816be519


In [3]:
rows = []
for line in open(RAW, encoding="utf-8"):
    p = json.loads(line)
    a = p["attributes"]
    rejection = a["rejection"] or {}
    rows.append({
        "id": p["id"],
        "source": p["source"],                          # Parliament: 2015-17, 2017-19, 2019-24, current
        "state": a["state"],
        "rejection_code": rejection.get("code"),
        "rejection_details": rejection.get("details"),
        "action": a["action"],
        "background": a["background"],
        "additional_details": a["additional_details"],
        "created_at": a["created_at"],
        "opened_at": a["opened_at"],
        "rejected_at": a["rejected_at"],
        "closed_at": a["closed_at"],
    })
raw = pd.DataFrame(rows)
for c in ["created_at", "opened_at", "rejected_at", "closed_at"]:
    raw[c] = pd.to_datetime(raw[c], utc=True)

assert len(raw) == raw.id.nunique() == 128_077
print(f"{len(raw):,} petitions")
raw.groupby(["source", "state"]).size().unstack(fill_value=0)

128,077 petitions


state,closed,open,rejected
source,,,
2015-17,10950,0,20780
2017-19,8152,0,19948
2019-24,13752,0,38280
current,5234,2293,8688


## 2. Labels (7 classes)

- state `open` or `closed` → **published**
- state `rejected` → the rejection code: duplicate, irrelevant, no-action, honours, already-happening, local
- **dropped:** fake-name (depends on the creator's name, which the model must not see — 51 identical texts were
  published under one name and rejected under another), operational (30) and foi (27), too rare to learn.

49 petitions were published first and rejected later. Their state is `rejected`, so they count as rejected.

In [4]:
raw["label"] = [make_label(s, c) for s, c in zip(raw.state, raw.rejection_code)]

in_scope = raw.label.notna()
print("Published first, rejected later:", (raw.state.eq("rejected") & raw.opened_at.notna()).sum())
print(f"Dropped (out of scope): {(~in_scope).sum():,} = {(~in_scope).mean():.1%}")
print(raw.loc[~in_scope, "rejection_code"].value_counts().to_string())
assert (~in_scope).sum() == 886

counts = raw.loc[in_scope, "label"].value_counts().reindex(CLASSES)
pd.DataFrame({"petitions": counts, "share": (counts / in_scope.sum()).round(3)})

Published first, rejected later: 49
Dropped (out of scope): 886 = 0.7%
rejection_code
fake-name      829
operational     30
foi             27


,petitions,share
label,,
published,40381,0.317
duplicate,44883,0.353
irrelevant,19347,0.152
no-action,16447,0.129
honours,3035,0.024
already-happening,2444,0.019
local,654,0.005


## 3. Decision time

The moment the moderator decided: `opened_at` for published petitions, `rejected_at` for rejected ones.
It is used for the time-based split (notebook 04) and for the 48-hour rule in step 5. It is never a model feature.

In [5]:
published = raw.state.isin(["open", "closed"])
raw["decision_time"] = raw.opened_at.where(published, raw.rejected_at)

assert raw.decision_time.notna().all()
print("Decision times from", raw.decision_time.min(), "to", raw.decision_time.max())

Decision times from 2015-07-20 11:05:07.590000+00:00 to 2026-09-25 16:55:20.794000+00:00


## 4. Text cleaning

`clean_text` in `src/preprocess.py` does, in order:

1. **Unescape HTML entities** (`&amp;` → `&`). There are none in this data, but new petitions could have them.
2. **Angle brackets:** 7 texts contain `<...>`. They are not real HTML tags but normal words in brackets
   (`<Daily Mail>`, `<ie. the UK gov.>`, `<http://...>`), so only the brackets are removed and the words are kept.
3. **URLs → `[URL]`.** A link is a long string of characters that carries no meaning for the models.
4. **Invisible characters** (control characters, zero-width spaces, byte-order marks) → space.
5. **Whitespace:** tabs, new lines and non-breaking spaces become one normal space.

Casing is **kept**, because the transformer uses it. TF-IDF and BM25 lower-case the text themselves later.

The model input is `text = action . background . additional_details` (an empty field is skipped, so there is no dangling " . ").

In [6]:
fields = ["action", "background", "additional_details"]
original = raw[fields].fillna("").agg(" ".join, axis=1)

has_angle = original.map(lambda s: bool(ANGLE_RE.search(s)))
print("Texts with HTML entities (&amp; ...):", original.str.contains(r"&[a-zA-Z]+;|&#\d+;").sum())
print("Texts with <...> brackets:", has_angle.sum())
print("Texts with URLs:", original.map(lambda s: bool(URL_RE.search(s))).sum())
print("Texts with U+FFFD (broken character, kept):", original.str.contains("�").sum())
print(f"Empty additional_details: {(raw.additional_details.fillna('').str.strip() == '').sum():,}"
      f" = {(raw.additional_details.fillna('').str.strip() == '').mean():.1%}")

print("\nThe 7 bracket cases, before -> after:")
for s in original[has_angle]:
    m = ANGLE_RE.search(s).group(0)
    print(f"  {m[:70]!r} -> {clean_text(m)[:70]!r}")

Texts with HTML entities (&amp; ...): 0
Texts with <...> brackets: 7


Texts with URLs: 6504
Texts with U+FFFD (broken character, kept): 2
Empty additional_details: 26,565 = 20.7%

The 7 bracket cases, before -> after:
  '<http://www.voteleavetakecontrol.org/why_vote_leave>' -> '[URL]'
  '<The Independent>' -> 'The Independent'
  '<www.uoguelph.ca>' -> '[URL]'
  '<Anna.Wright@slough.gov.uk>' -> 'Anna.Wright@slough.gov.uk'
  '<ie. the UK gov.>' -> 'ie. the UK gov.'
  '<https://www.theguardian.com/uk-news/2020/nov/02/homelessness-charitie' -> '[URL]'
  '<https://commonslibrary.parliament.uk/tag/house-of-commons-trends//>' -> '[URL]'


In [7]:
# build the model text from the original fields first, then clean the separate fields
raw["text"] = [build_text(a, b, c) for a, b, c in zip(raw.action, raw.background, raw.additional_details)]
for f in fields:
    raw[f] = raw[f].map(clean_text)

assert raw.text.str.len().gt(0).all()
assert not raw.text.map(lambda s: bool(ANGLE_RE.search(s) or URL_RE.search(s))).any()
n_url = raw.text.str.contains("[URL]", regex=False).sum()
assert n_url == 6_504
print(f"Texts with [URL]: {n_url:,}")
print("Words per text:", raw.text.str.split().str.len().describe()[["min", "50%", "max"]].to_dict())

example = raw.text[raw.text.str.contains("[URL]", regex=False)].iloc[0]
print("\nExample:", example[:400])

Texts with [URL]: 6,504


Words per text: {'min': 5.0, '50%': 105.0, 'max': 228.0}

Example: Stop all immigration and close the UK borders until ISIS is defeated. . In February 2015 Dr Shea, Nato’s Deputy Assistant Secretary General for Emerging Security Challenges, warned there would be IS jihadists on the refugee boats. IS also threatened to flood Europe with 500,000 jihadists. . Allowing uncontrolled immigration and taking in these refugees potentially endangers the entire UK populatio


**Likely non-English texts.** The Milestone-1 audit used a rough rule (few of 20 very common English words such as
"the", "of") and found 76. Looking at them, almost all were short English petitions such as
"Make trees . Make trees . Make trees", which simply do not use those 20 words.

Better rule: the collection is almost entirely English, so English words are *common in it*. A text is flagged when
fewer than 30% of its words appear in at least 20 petitions (and it has at least 3 words). A word here is a run of letters
in any alphabet, so Arabic counts too. Flagged texts are **kept** (real petitions with real decisions) and only marked,
so the EDA and the error analysis can report them.

In [8]:
words = raw.text.str.replace("[URL]", " ", regex=False).str.lower().str.findall(r"[^\W\d_]+")
petitions_per_word = collections.Counter(w for ws in words for w in set(ws))   # each word once per petition
common = {w for w, n in petitions_per_word.items() if n >= 20}
common_share = words.map(lambda ws: sum(w in common for w in ws) / max(len(ws), 1))
raw["likely_non_english"] = (common_share < 0.3) & (words.str.len() >= 3)

print("Likely non-English:", raw.likely_non_english.sum())
raw.loc[raw.likely_non_english, ["id", "label", "text"]].assign(text=lambda t: t.text.str[:110])

Likely non-English: 5


,id,label,text
19185,124327,irrelevant,Ce que j'aimerais que la Ville fasse... . Ville de Montréal tienne une consultation publique sur...
61102,589459,no-action,عدن تحت الحصار بقطع الخدمات عن اهلها . نريد من حكومة بريطانيا التدخل السريع بمناشدة الحكومة اليم...
69245,587745,irrelevant,We oppose the 'central and eastern river crossing' for Harlow and Gilston Town. . T𝐡𝐞 𝐩𝐫𝐨𝐩𝐨𝐬𝐚𝐥 𝐩...
75961,619050,irrelevant,"Change national anthem . Replace with the prodigy pendulum remix . Du, Du du du du, Du du du du,..."
107095,240291,no-action,Artem shypulya vivre et laisser vivre . vivre et laisser vivre . vivre et laisser vivre


## 5. Gold duplicate pairs (for retrieval)

When moderators reject a petition as a duplicate, the rejection message links the open petition(s) it duplicates.
These links are the "correct answers" for retrieval. Each petition ID is read with the pattern `petitions/+(\d+)`
(the `+` also catches 30 broken links written `petitions//ID`). A link is **removed** if:

1. it points to the petition itself;
2. the target is not in the data;
3. the target was rejected — the candidate pool only contains published petitions;
4. the target was opened more than 48 hours after the duplicate was rejected. (Opened within 48 h is kept: the moderator
   linked a petition that was about to be published.)

All remaining links are kept. If a duplicate has several, any of them counts as a hit.
Duplicates without any valid link stay in the classification data and are only left out of the retrieval evaluation.

*Note:* the Milestone-1 audit counted the first link of each petition only. Here every link is checked, so the
per-link counts differ a little, but the number of duplicates without a valid link (179) is the same.

In [9]:
info = raw.set_index("id")
dups = raw[raw.label.eq("duplicate")]
n_links = dups.rejection_details.map(lambda d: len(linked_ids(d)))

pairs = []
removed = {"self-link": 0, "target not in data": 0, "target rejected": 0, "target opened > 48 h later": 0}
for qid, details, t in zip(dups.id, dups.rejection_details, dups.decision_time):
    for tid in linked_ids(details):
        if tid == qid:
            removed["self-link"] += 1
        elif tid not in info.index:
            removed["target not in data"] += 1
        elif info.at[tid, "state"] == "rejected":
            removed["target rejected"] += 1
        elif info.at[tid, "opened_at"] > t + pd.Timedelta(hours=48):
            removed["target opened > 48 h later"] += 1
        else:
            pairs.append((qid, tid))
gold = pd.DataFrame(pairs, columns=["query_id", "target_id"])
per_query = gold.groupby("query_id").size()

broken = dups.rejection_details.str.contains(r"petitions//\d", na=False).sum()
print(f"Duplicate rejections: {len(dups):,}")
print(f"  without any link: {(n_links == 0).sum()}")
print(f"  with a broken petitions//ID link: {broken}")
print(f"Links found: {n_links.sum():,}")
for reason, n in removed.items():
    print(f"  removed, {reason}: {n}")
print(f"Valid links = gold pairs: {len(gold):,}")
late = info.loc[gold.target_id, "opened_at"].values > info.loc[gold.query_id, "decision_time"].values
print(f"  of which the target opened after the rejection, within 48 h: {late.sum()}")
print(f"Duplicates with >= 1 valid link (retrieval queries): {len(per_query):,}")
print(f"  with several valid links: {(per_query > 1).sum():,}")
print(f"Duplicates with no valid link (left out of retrieval evaluation): {len(dups) - len(per_query)}")
assert len(dups) - len(per_query) == 179

other = raw.rejection_code.notna() & raw.rejection_code.ne("duplicate")
print(f"\nNon-duplicate rejections that also link a petition ('you may wish to sign ...'),"
      f" NOT gold pairs: {raw.loc[other, 'rejection_details'].map(lambda d: len(linked_ids(d)) > 0).sum():,}")

Duplicate rejections: 44,883
  without any link: 82
  with a broken petitions//ID link: 30
Links found: 61,287
  removed, self-link: 6
  removed, target not in data: 47
  removed, target rejected: 63
  removed, target opened > 48 h later: 39
Valid links = gold pairs: 61,132
  of which the target opened after the rejection, within 48 h: 183
Duplicates with >= 1 valid link (retrieval queries): 44,704
  with several valid links: 6,833
Duplicates with no valid link (left out of retrieval evaluation): 179

Non-duplicate rejections that also link a petition ('you may wish to sign ...'), NOT gold pairs: 8,015


Some gold pairs share almost no words — a paraphrase, not a copy. This is why dense embeddings are tried after BM25.

In [10]:
show = gold[gold.query_id.isin([150557, 177199, 130881])]
show.assign(query_title=info.loc[show.query_id, "action"].values,
            target_title=info.loc[show.target_id, "action"].values)

,query_id,target_id,query_title,target_title
634,177199,146612,Stop fake news and make it illegal to knowingly spread them,"Criminalise politicians and the media for lying, misleading or misinforming."
1566,150557,121973,Ban the t.v licence,Stop TV licensing!
2432,130881,121701,Make it illegal for politicians to have other employment & or business interests,That MPs should work full time as an MP and not have outside jobs.


## 6. Leakage test

Two layers:

- **(a)** The saved tables contain only the allow-listed columns (`CLEAN_COLUMNS` in `src/preprocess.py`).
- **(b)** Every model notebook calls `leakage_check(X)` on exactly what it is about to give the model.
  It stops with an error if `X` contains anything except `text`. Below it is shown passing for the text and failing
  for other columns, including `opened_at`: in the raw data it is set for 100% of published petitions and 0.1% of rejected ones,
  so a model given it would look almost perfect while learning nothing about the text.

Out-of-scope petitions keep their real code (fake-name, operational, foi) in the `label` column.

In [11]:
raw["label"] = raw.label.fillna(raw.rejection_code)
clean = raw.loc[in_scope, CLEAN_COLUMNS].reset_index(drop=True)
out_of_scope = raw.loc[~in_scope, CLEAN_COLUMNS].reset_index(drop=True)

# (a) only allow-listed columns in the saved tables
leakage_check(clean, allowed=CLEAN_COLUMNS)
leakage_check(out_of_scope, allowed=CLEAN_COLUMNS)
leakage_check(gold, allowed=["query_id", "target_id"])
print("(a) saved columns:", list(clean.columns))

# (b) the model input check passes for the text ...
leakage_check(clean[MODEL_INPUTS])
print("(b) text only -> passes")

# ... and blocks any other column
toy = pd.DataFrame({"signature_count": [10]})
for X in [clean[["text", "opened_at"]], clean[["text", "label"]], clean[["text", "decision_time"]], toy]:
    try:
        leakage_check(X)
        raise AssertionError("the leakage check let a forbidden column through")
    except ValueError as e:
        print("(b) blocked:", e)

(a) saved columns: ['id', 'source', 'label', 'action', 'background', 'additional_details', 'text', 'created_at', 'opened_at', 'rejected_at', 'closed_at', 'decision_time', 'likely_non_english']
(b) text only -> passes
(b) blocked: Leakage check failed, these columns are not allowed: ['opened_at']
(b) blocked: Leakage check failed, these columns are not allowed: ['label']
(b) blocked: Leakage check failed, these columns are not allowed: ['decision_time']
(b) blocked: Leakage check failed, these columns are not allowed: ['signature_count']


## 7. Save

In [12]:
clean.to_parquet(OUT / "petitions.parquet", index=False)
out_of_scope.to_parquet(OUT / "out_of_scope.parquet", index=False)
gold.to_parquet(OUT / "gold_pairs.parquet", index=False)

assert len(clean) + len(out_of_scope) == 128_077
for f in sorted(OUT.glob("*.parquet")):
    print(f"{f.name:22s} {len(pd.read_parquet(f)):>9,} rows  {f.stat().st_size / 1e6:6.1f} MB")

gold_pairs.parquet        61,132 rows     0.5 MB
out_of_scope.parquet         886 rows     0.6 MB


petitions.parquet        127,191 rows    94.2 MB


## Summary

- 128,077 petitions → **127,191** in 7 classes + **886** out-of-scope (fake-name 829, operational 30, foi 27).
- Text cleaned: 6,504 texts had URLs (→ `[URL]`), 7 had words in angle brackets (brackets removed); casing kept.
- Only a handful of texts are likely non-English (flagged, kept); the Milestone-1 figure of 76 was an overestimate.
- **44,704** duplicates have at least one valid gold link (61,132 pairs); 179 have none and are left out of the retrieval evaluation only.
- Leakage test: only `text` can reach a model; the check is reused in every model notebook.

Next: `04_split.ipynb` — train / validation / test by decision time, and the time-aware candidate pools.